In [ ]:
!pip install transformers torch accelerate huggingface_hub


In [ ]:
from huggingface_hub import login
from google.colab import userdata
login(userdata.get('HF_TOKEN'))

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "issai/LLama-3.1-KazLLM-1.0-8B"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Model loaded!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/987 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/155 [00:00<?, ?B/s]

Model loaded!


In [ ]:
def ask_kazllm(problem_text):
    messages = [
        {"role": "system", "content": "You are a physics teacher. Solve the given physics problem step by step. Write all math using plain text, not LaTeX."},
        {"role": "user", "content": problem_text}
    ]

    input_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=1000,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return response

print("Function ready!")

Function ready!


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving problems.xlsx to problems.xlsx


In [ ]:
import pandas as pd
import time

df_problems = pd.read_excel('problems.xlsx')

results = []

for _, row in df_problems.iterrows():
    problem_id = row['Problem_ID']

    for language in ['English', 'Kazakh']:
        problem_text = row['Eng_version'] if language == 'English' else row['Kaz_version']

        if pd.isna(problem_text) or str(problem_text).strip() == '':
            continue

        print(f"Problem {problem_id} | {language}")

        try:
            answer = ask_kazllm(problem_text)
            print(f"  KazLLM: done")
        except Exception as e:
            answer = f"ERROR: {e}"
            print(f"  KazLLM: error - {e}")

        results.append({
            'Problem_ID': problem_id,
            'Topic': row['Topic'],
            'Level': row['Level'],
            'Language': language,
            'Model': 'KazLLM-8B',
            'Model_Answer': answer,
            'Accuracy (0-2)': '',
            'Explanation_Quality (1-5)': '',
            'Language_Performance (1-5)': '',
            'Evaluator': '',
            'Notes': ''
        })

        time.sleep(1)


df_kazllm = pd.DataFrame(results)
df_kazllm.to_excel('kazllm_results.xlsx', index=False)
print("Done! Saved to kazllm_results.xlsx")

Problem 1 | English
  KazLLM: done
Problem 1 | Kazakh
  KazLLM: done
Problem 2 | English
  KazLLM: done
Problem 2 | Kazakh
  KazLLM: done
Problem 3 | English
  KazLLM: done
Problem 3 | Kazakh
  KazLLM: done
Problem 4 | English
  KazLLM: done
Problem 4 | Kazakh
  KazLLM: done
Problem 5 | English
  KazLLM: done
Problem 5 | Kazakh
  KazLLM: done
Problem 6 | English
  KazLLM: done
Problem 6 | Kazakh
  KazLLM: done
Problem 7 | English
  KazLLM: done
Problem 7 | Kazakh
  KazLLM: done
Problem 8 | English
  KazLLM: done
Problem 8 | Kazakh
  KazLLM: done
Problem 9 | English
  KazLLM: done
Problem 9 | Kazakh
  KazLLM: done
Problem 10 | English
  KazLLM: done
Problem 10 | Kazakh
  KazLLM: done
Problem 11 | English
  KazLLM: done
Problem 11 | Kazakh
  KazLLM: done
Problem 12 | English
  KazLLM: done
Problem 12 | Kazakh
  KazLLM: done
Problem 13 | English
  KazLLM: done
Problem 13 | Kazakh
  KazLLM: done
Problem 14 | English
  KazLLM: done
Problem 14 | Kazakh
  KazLLM: done
Problem 15 | English
  K

KeyboardInterrupt: 

In [ ]:
from google.colab import files
df_kazllm = pd.DataFrame(results)
df_kazllm.to_excel('kazllm_results.xlsx', index=False)
files.download('kazllm_results.xlsx')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:

import pandas as pd
import time
from google.colab import files
df_problems = pd.read_excel('problems.xlsx')
print(df_problems.columns.tolist())
df_problems.head()
b_problems = df_problems[df_problems['Level'] == 'B'].head(10)
c_problems = df_problems[df_problems['Level'] == 'C'].head(10)
df_subset = pd.concat([b_problems, c_problems])

results_bc = []

for _, row in df_subset.iterrows():
    problem_id = row['Problem_ID']

    for language in ['English', 'Kazakh']:
        problem_text = row['Eng_version'] if language == 'English' else row['Kaz_version']

        if pd.isna(problem_text) or str(problem_text).strip() == '':
            continue

        print(f"Problem {problem_id} | {language}")

        try:
            answer = ask_kazllm(problem_text)
            print(f"  KazLLM: done")
        except Exception as e:
            answer = f"ERROR: {e}"
            print(f"  KazLLM: error - {e}")

        results_bc.append({
            'Problem_ID': problem_id,
            'Topic': row['Topic'],
            'Level': row['Level'],
            'Language': language,
            'Model': 'KazLLM-8B',
            'Model_Answer': answer,
            'Accuracy (0-2)': '',
            'Explanation_Quality (1-5)': '',
            'Language_Performance (1-5)': '',
            'Evaluator': '',
            'Notes': ''
        })

        time.sleep(1)

df_bc = pd.DataFrame(results_bc)
df_bc.to_excel('kazllm_results_bc.xlsx', index=False)
from google.colab import files
files.download('kazllm_results_bc.xlsx')
print("Done!")

['Problem_ID', 'Topic', 'Level', 'Kaz_version', 'Eng_version', 'Correct_answer']
Problem 31 | English
  KazLLM: done
Problem 31 | Kazakh
  KazLLM: done
Problem 32 | English
  KazLLM: done
Problem 32 | Kazakh
  KazLLM: done
Problem 33 | English
  KazLLM: done
Problem 33 | Kazakh
  KazLLM: done
Problem 34 | English
  KazLLM: done
Problem 34 | Kazakh
  KazLLM: done
Problem 35 | English
  KazLLM: done
Problem 35 | Kazakh
  KazLLM: done
Problem 36 | English
  KazLLM: done
Problem 36 | Kazakh
  KazLLM: done
Problem 37 | English
  KazLLM: done
Problem 37 | Kazakh
  KazLLM: done
Problem 38 | English


KeyboardInterrupt: 

In [ ]:
import pandas as pd
from google.colab import files
df_bc = pd.DataFrame(results_bc)
df_bc.to_excel('kazllm_results_bc.xlsx', index=False)
files.download('kazllm_results_bc.xlsx')



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
import time
from google.colab import files

# Загружаем problems
uploaded = files.upload()
df_problems = pd.read_excel('problems.xlsx')

# Только B уровень 38-40
df_bc = df_problems[
    (df_problems['Level'] == 'B') &
    (df_problems['Problem_ID'].isin([38, 39, 40]))
]

results_kaz = []

for _, row in df_bc.iterrows():
    problem_id = row['Problem_ID']

    for language in ['English', 'Kazakh']:
        problem_text = row['Eng_version'] if language == 'English' else row['Kaz_version']

        if pd.isna(problem_text) or str(problem_text).strip() == '':
            continue

        print(f"Problem {problem_id} | {language}")

        try:
            answer = ask_kazllm(problem_text)
            print(f"  KazLLM: done")
        except Exception as e:
            answer = f"ERROR: {e}"
            print(f"  KazLLM: error - {e}")

        results_kaz.append({
            'Problem_ID': problem_id,
            'Topic': row['Topic'],
            'Level': row['Level'],
            'Language': language,
            'Model': 'KazLLM-8B',
            'Model_Answer': answer,
            'Accuracy (0-2)': '',
            'Explanation_Quality (1-5)': '',
            'Language_Performance (1-5)': '',
            'Evaluator': '',
            'Notes': ''
        })

        time.sleep(1)

    # Сохраняем после каждой задачи
    pd.DataFrame(results_kaz).to_excel('kazllm_b_38_40.xlsx', index=False)
    print(f"Auto-saved after problem {problem_id}")

# Скачиваем
from google.colab import files
files.download('kazllm_b_38_40.xlsx')
print("Done!")

Saving problems.xlsx to problems (1).xlsx
Problem 38 | English
  KazLLM: done
Problem 38 | Kazakh
  KazLLM: done
Auto-saved after problem 38
Problem 39 | English
  KazLLM: done
Problem 39 | Kazakh
  KazLLM: done
Auto-saved after problem 39
Problem 40 | English
  KazLLM: done
Problem 40 | Kazakh
  KazLLM: done
Auto-saved after problem 40


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Done!


In [ ]:
import pandas as pd
import time
from google.colab import files

# Upload problems
uploaded = files.upload()
df_problems = pd.read_excel('problems.xlsx')

# Only C level, first 10 problems
df_c = df_problems[df_problems['Level'] == 'C'].head(10)

results_kaz = []

for _, row in df_c.iterrows():
    problem_id = row['Problem_ID']
    for language in ['English', 'Kazakh']:
        problem_text = row['Eng_version'] if language == 'English' else row['Kaz_version']

        if pd.isna(problem_text) or str(problem_text).strip() == '':
            continue

        print(f"Problem {problem_id} | {language}")

        try:
            answer = ask_kazllm(problem_text)
            print(f"  KazLLM: done")
        except Exception as e:
            answer = f"ERROR: {e}"
            print(f"  KazLLM: error - {e}")

        results_kaz.append({
            'Problem_ID': problem_id,
            'Topic': row['Topic'],
            'Level': row['Level'],
            'Language': language,
            'Model': 'KazLLM-8B',
            'Model_Answer': answer,
            'Accuracy (0-2)': '',
            'Explanation_Quality (1-5)': '',
            'Language_Performance (1-5)': '',
            'Evaluator': '',
            'Notes': ''
        })
        time.sleep(1)

    # Auto-save after each problem
    pd.DataFrame(results_kaz).to_excel('kazllm_c_10.xlsx', index=False)
    print(f"Auto-saved after problem {problem_id}")

# Download
files.download('kazllm_c_10.xlsx')
print("Done!")

Saving problems.xlsx to problems (1).xlsx
Problem 43 | English
